# MCfd refactored validation suite

这个 notebook 是根据你上传的 `MCfd.ipynb` 重新整理的版本。保留原 notebook 中的四个估计器：MC-I、MC-II、GJ-I、GJ-II，并把实验系统化为三组：

1. 不同 $\alpha$ 的影响，尤其是 $\alpha\to2$；
2. $M$ 对误差的影响，并诊断 GJ 在大 $M$ 时误差增大的原因；
3. 非光滑函数 $f(t)=(t-t_c)_+^\beta$ 下的表现。

核心函数都放在 `MCfd_refactored_validation.py` 中；这个 notebook 负责调用和展示。

In [ ]:
from pathlib import Path
import json
import sys

# Keep this notebook and MCfd_refactored_validation.py in the same folder.
sys.path.insert(0, str(Path.cwd()))
from MCfd_refactored_validation import *

set_nature_style()
outdir = Path('MCfd_refactored_outputs')
outdir.mkdir(exist_ok=True)
seed = 229

## 1. $\alpha$ sweep for the smooth benchmark

这里使用原 notebook 的光滑函数

$$
f(t)=e^{-t},\qquad t=1.5.
$$

Caputo 真值仍对应 notebook 中的

$$
D_t^\alpha e^{\lambda t}=\lambda^2 t^{2-\alpha}E_{1,3-\alpha}(\lambda t).
$$

代码中用等价的 Kummer hypergeometric 表达式计算，避免依赖 `pymittagleffler`。

In [ ]:
alpha_result = experiment_alpha_sweep(outdir, seed=seed)
print('saved:', outdir / 'fig01_alpha_sweep_exp.png')

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(outdir / 'fig01_alpha_sweep_exp.png')))

## 2. $M$ sweep and GJ endpoint diagnosis

这部分对应你原 notebook 的 `different M` 段。除了原始 raw quotient，还加入了 exponential benchmark 下的 stabilized quotient。

原因是 GJ 节点映射到 $[0,1]$ 后，最近端点的节点满足大约

$$
\tau_{\min}\sim M^{-2}.
$$

Type-II 中出现

$$
\frac{f(t)-f(t-t\tau)-t\tau f'(t)}{(t\tau)^2},
$$

连续层面是 removable singularity，但 raw floating-point 计算会发生 cancellation，再被 $(t\tau)^{-2}$ 放大。

In [ ]:
M_result = experiment_M_sweep(outdir, seed=seed, mc_repeats=8)
print('saved:', outdir / 'fig02_M_sweep_exp_diagnostic.png')

In [ ]:
display(Image(filename=str(outdir / 'fig02_M_sweep_exp_diagnostic.png')))

## 3. Non-smooth benchmark

非光滑测试函数取

$$
f(t)=(t-t_c)_+^\beta,\qquad 1<\beta<2.
$$

它是 $C^1$ 但不是 $C^2$。解析 Caputo 导数为

$$
D_t^\alpha (t-t_c)_+^\beta
=\frac{\Gamma(\beta+1)}{\Gamma(\beta+1-\alpha)}(t-t_c)_+^{\beta-\alpha}.
$$

这可以直接检验理论中的光滑性假设：当 memory interval 内部出现 kink，GJ 的高阶/谱收敛会降级，并且误差通常不再单调。

In [ ]:
nonsmooth_result = experiment_nonsmooth(outdir, seed=seed)
print('saved:', outdir / 'fig03_nonsmooth_tests.png')

In [ ]:
display(Image(filename=str(outdir / 'fig03_nonsmooth_tests.png')))

## Save JSON results

如果要在论文里生成表格或进一步处理误差数据，可以直接读取 `MCfd_refactored_results.json`。

In [ ]:
results = {
    'alpha_sweep_exp': alpha_result,
    'M_sweep_exp_diagnostic': M_result,
    'nonsmooth_tests': nonsmooth_result,
}
with open(outdir / 'MCfd_refactored_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)
print(outdir / 'MCfd_refactored_results.json')